Step L1. Notebook to join liu greenness to (GLAKES) and compute LEV measures

Prerequisite: fix GLAKES geoms, clip to desired domain

"""Originally did this with QGIS"""

In [1]:
import pandas as pd
from pathlib import Path
import geopandas as gpd
import seaborn as sns
from matplotlib import pyplot as plt
import os
import numpy as np
from shapely.geometry import box

from land_cover.load import GLAKES_MA_pth, GLAKES_NDVI_pth, GLAKES_VO_pth, loadGLAKES, GLAKES_filtered_fix_aqveg_pth

%load_ext autoreload
%autoreload 2

In [2]:
pth_glakes_ts = "/Volumes/metis/Datasets/GLAKES/GLAKES area time series.csv"
gdf_ts = pd.read_csv(pth_glakes_ts).rename(columns={"Lake_id": "Lake_id_glakes"})
gdf_ts

,Lake_id_glakes,area_1984_1999_wm,area_2000_2009_wm,area_2010_2019_wm,area_1984_1999_nm,area_2000_2009_nm,area_2010_2019_nm
0,1.0,373104.607033,375634.277372,372483.408836,373108.344879,375636.918257,372486.611142
1,2.0,115531.989587,116355.845745,115948.199697,115556.876078,116370.025386,115966.096349
2,3.0,80903.091562,81679.557074,81192.035002,80913.627201,81688.666172,81201.706214
3,4.0,12597.283530,12704.303600,12698.317374,65501.912000,66385.510155,66255.650234
4,5.0,37599.512076,20814.265397,13481.348368,37599.567925,20814.664144,13481.681372
...,...,...,...,...,...,...,...
3423906,3426385.0,0.012933,0.000000,0.000056,0.012814,0.000000,0.000032
3423907,3426386.0,0.025496,0.026948,0.027198,0.026006,0.027360,0.027505
3423908,3426387.0,0.026927,0.022759,0.025122,0.027298,0.023162,0.025541
3423909,3426388.0,0.025455,0.028398,0.027267,0.026419,0.029254,0.028069


In [3]:
df_csv_MA = pd.read_csv(GLAKES_MA_pth).rename(columns={"Lake_id": "Lake_id_glakes"})
df_csv_NDVI = pd.read_csv(GLAKES_NDVI_pth).rename(columns={"Lake_id": "Lake_id_glakes"})
df_csv_VO = pd.read_csv(GLAKES_VO_pth).rename(columns={"Lake_id": "Lake_id_glakes"})
df_csv_VO.head()

,Lake_id_glakes,Lat,Lon,vegeP1,validP1,vegeP2,validP2,vegeP3,validP3,occP1,occP2,occP3
0,16,60.829856,31.477989,31487.0,558406049.0,43088.0,487100474.0,77380.0,508681505.0,0.005639,0.008846,0.015212
1,58,58.547195,27.544395,38675.0,99463380.0,70719.0,97606762.0,143783.0,96193435.0,0.038884,0.072453,0.149473
2,106,58.327570,14.536565,750.0,46520588.0,733.0,54741173.0,1265.0,49550497.0,0.001612,0.001339,0.002553
3,143,45.729056,34.919955,45.0,40367266.0,322.0,32679823.0,448.0,27144461.0,0.000111,0.000985,0.001650
4,155,63.581397,34.753564,709.0,34316772.0,808.0,18043549.0,2580.0,28927685.0,0.002066,0.004478,0.008919


In [4]:
df_csv_MA.head()

,Lake_id_glakes,Lat,Lon,areaP1,areaP2,areaP3
0,16,60.829856,31.477989,19.471456,14.927326,15.689287
1,58,58.547195,27.544395,18.546118,16.876630,20.688981
2,106,58.327570,14.536565,1.131301,0.650710,0.982445
3,143,45.729056,34.919955,0.135702,0.322291,0.503227
4,155,63.581397,34.753564,1.131626,1.019905,2.147928


In [5]:
# Confirm that occurence is just veg / valid ✅
(df_csv_VO.vegeP1 / df_csv_VO.validP1).head()


0    0.000056
1    0.000389
2    0.000016
3    0.000001
4    0.000021
dtype: float64

In [6]:
# GLAKES spatial attributes in NA and Scandinavia
gdf_glakes = loadGLAKES()

In [7]:
df_csv_MA = df_csv_MA.merge(
    df_csv_NDVI[["Lake_id_glakes", "NDVI8499", "NDVI0010", "NDVI1121"]],
    on=["Lake_id_glakes"],
    how="outer",
)
df_csv_MA = df_csv_MA.merge(
    df_csv_VO[["Lake_id_glakes", "occP1", "occP2", "occP3"]], on=["Lake_id_glakes"], how="outer"
)

# join in df_csv_MA/NDVI to get greenness
gdf = gdf_glakes.merge(
    df_csv_MA.drop(columns=["Lat", "Lon"]),
    on="Lake_id_glakes",
    how="left", # keep all lakes in study area, even though some may not have greenness for some reason (area = 0?)
)
# Join in time series, keep _wm, not _nm
gdf = gdf.merge(
    gdf_ts.drop(columns=["area_1984_1999_nm", "area_2000_2009_nm", "area_2010_2019_nm"]),
    on="Lake_id_glakes",
    how="left",
).drop(columns="OBJECTID_glakes")
gdf.head()

,Lake_id_glakes,Area_bound_glakes,Area_PW_glakes,Continent_glakes,Lat_glakes,Lon_glakes,GFed_flag_glakes,PFed_flag_glakes,Endo_flag_glakes,Rser_flag_glakes,...,areaP3,NDVI8499,NDVI0010,NDVI1121,occP1,occP2,occP3,area_1984_1999_wm,area_2000_2009_wm,area_2010_2019_wm
0,3,82155.420436,79514.496612,North America,47.526368,-87.757371,0,0,0,0,...,6.284501,0.64,0.66,0.68,0.000651,0.000717,0.001467,80903.091562,81679.557074,81192.035002
1,8,30657.104741,28864.201812,North America,65.998658,-120.968462,0,1,0,0,...,0.978539,0.53,0.55,0.58,0.000011,0.000035,0.000098,30156.030260,30565.860504,30502.390581
2,10,27480.339361,26652.927111,North America,52.825958,-97.738194,0,1,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26977.563484,27182.922134,27256.549512
3,11,26827.549399,25926.211686,North America,61.769419,-113.811408,0,1,0,0,...,24.076995,0.65,0.68,0.70,0.002910,0.003569,0.008582,26559.357871,26710.565325,26650.593451
4,16,17460.818811,16808.357727,Europe,60.829856,31.477989,0,0,0,0,...,15.689287,0.68,0.70,0.72,0.005639,0.008846,0.015212,17171.066362,17284.003108,17304.408409


In [8]:
gdf.info(show_counts=True)

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1878284 entries, 0 to 1878283
Data columns (total 25 columns):
 #   Column             Non-Null Count    Dtype   
---  ------             --------------    -----   
 0   Lake_id_glakes     1878284 non-null  int64   
 1   Area_bound_glakes  1878284 non-null  float64 
 2   Area_PW_glakes     1877185 non-null  float64 
 3   Continent_glakes   1878284 non-null  object  
 4   Lat_glakes         1878284 non-null  float64 
 5   Lon_glakes         1878284 non-null  float64 
 6   GFed_flag_glakes   1878284 non-null  int64   
 7   PFed_flag_glakes   1878284 non-null  int64   
 8   Endo_flag_glakes   1878284 non-null  int64   
 9   Rser_flag_glakes   1878284 non-null  int64   
 10  Shape_Leng_glakes  1878284 non-null  float64 
 11  Shape_Area_glakes  1878284 non-null  float64 
 12  geometry           1878284 non-null  geometry
 13  areaP1             542298 non-null   float64 
 14  areaP2             542944 non-null   float64 
 15  areaP3 

Note: some lakes even in N domain are missing aquatic veg estimates for some reason

In [9]:
df_csv_MA.query("Lake_id_glakes == 1005155")
gdf_glakes.query("Lake_id_glakes == 1005155")

,OBJECTID_glakes,Lake_id_glakes,Area_bound_glakes,Area_PW_glakes,Continent_glakes,Lat_glakes,Lon_glakes,GFed_flag_glakes,PFed_flag_glakes,Endo_flag_glakes,Rser_flag_glakes,Shape_Leng_glakes,Shape_Area_glakes,geometry
542082,1005155.0,1005155,0.166687,0.139891,North America,64.004207,-109.24159,0,1,0,0,0.03,0.000031,"POLYGON ((-109.23875 64.00725, -109.23875 64.0..."


In [10]:
len(df_csv_NDVI)

321989

In [11]:
len(df_csv_MA)

1255284

In [12]:
assert gdf["geometry"].notnull().all(), "Some geometry values in gdf are empty"
assert gdf_glakes["geometry"].notnull().all(), "Some geometry values in gdf_glakes are empty"

In [13]:
# Add LEV stats
gdf["LEV_p1"] = gdf.areaP1 / gdf["area_1984_1999_wm"] * 100
gdf["LEV_p2"] = gdf.areaP2 / gdf["area_2000_2009_wm"] * 100
gdf["LEV_p3"] = gdf.areaP3 / gdf["area_2010_2019_wm"] * 100

for i in range(1, 4):
    gdf.loc[np.isinf(gdf[f"LEV_p{i}"]), f"LEV_p{i}"] = np.nan

gdf["LEV_p13ain"] = gdf.areaP3 - gdf.areaP1 # "LEV period 1 to period 3 absolute increase"
gdf["LEV_p13rin"] = gdf.LEV_p3 - gdf.LEV_p1 # "LEV period 1 to period 3 relative increase"
gdf["LEV_p23ain"] = gdf.areaP3 - gdf.areaP2  # "LEV period 2 to period 3 absolute increase"
gdf["LEV_p23rin"] = gdf.LEV_p3 - gdf.LEV_p2  # "LEV period 2 to period 3 relative increase"

gdf["LEV_p13occ_ain"] = gdf.occP3 - gdf.occP1  # "LEV period 1 to period 3 absolute increase"
gdf["LEV_p23occ_ain"] = gdf.occP3 - gdf.occP2  # "LEV period 2 to period 3 absolute increase"

gdf["LEV_p13in"] = (
    gdf["LEV_p13ain"] / gdf["area_1984_1999_wm"] * 100
)  # "Another type of relative increase, normalized to initial lake area

In [14]:
gdf.head()

,Lake_id_glakes,Area_bound_glakes,Area_PW_glakes,Continent_glakes,Lat_glakes,Lon_glakes,GFed_flag_glakes,PFed_flag_glakes,Endo_flag_glakes,Rser_flag_glakes,...,LEV_p1,LEV_p2,LEV_p3,LEV_p13ain,LEV_p13rin,LEV_p23ain,LEV_p23rin,LEV_p13occ_ain,LEV_p23occ_ain,LEV_p13in
0,3,82155.420436,79514.496612,North America,47.526368,-87.757371,0,0,0,0,...,0.008512,0.007922,0.007740,-0.601649,-0.000771,-0.185964,-0.000181,0.000816,0.000750,-0.000744
1,8,30657.104741,28864.201812,North America,65.998658,-120.968462,0,1,0,0,...,0.001103,0.002080,0.003208,0.645770,0.002105,0.342653,0.001128,0.000087,0.000064,0.002141
2,10,27480.339361,26652.927111,North America,52.825958,-97.738194,0,1,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,11,26827.549399,25926.211686,North America,61.769419,-113.811408,0,1,0,0,...,0.048299,0.055355,0.090343,11.249214,0.042045,9.291329,0.034988,0.005672,0.005013,0.042355
4,16,17460.818811,16808.357727,Europe,60.829856,31.477989,0,0,0,0,...,0.113397,0.086365,0.090666,-3.782168,-0.022730,0.761961,0.004301,0.009573,0.006366,-0.022026


In [15]:
gdf.info(show_counts=True)

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1878284 entries, 0 to 1878283
Data columns (total 35 columns):
 #   Column             Non-Null Count    Dtype   
---  ------             --------------    -----   
 0   Lake_id_glakes     1878284 non-null  int64   
 1   Area_bound_glakes  1878284 non-null  float64 
 2   Area_PW_glakes     1877185 non-null  float64 
 3   Continent_glakes   1878284 non-null  object  
 4   Lat_glakes         1878284 non-null  float64 
 5   Lon_glakes         1878284 non-null  float64 
 6   GFed_flag_glakes   1878284 non-null  int64   
 7   PFed_flag_glakes   1878284 non-null  int64   
 8   Endo_flag_glakes   1878284 non-null  int64   
 9   Rser_flag_glakes   1878284 non-null  int64   
 10  Shape_Leng_glakes  1878284 non-null  float64 
 11  Shape_Area_glakes  1878284 non-null  float64 
 12  geometry           1878284 non-null  geometry
 13  areaP1             542298 non-null   float64 
 14  areaP2             542944 non-null   float64 
 15  areaP3 

In [16]:
# write out
gdf.to_file(GLAKES_filtered_fix_aqveg_pth)
print(f"Wrote: {GLAKES_filtered_fix_aqveg_pth}")

Wrote: /Volumes/metis/Datasets/Liu_aq_veg/figshare/v4/25012091/edk_out/GLAKES_filtered_fix_aqveg.gpkg
